## Проект визуализации сети поставщиков

Цель - визуально отразить сеть поставщиков компании по какой-то группе МТР и увидеть тренды, не замечаемые до этого. При наведении на вершины графа выводится основная информация по компании. Величина вершины регулируется общей суммой, выигранной при участии во всех отборах. Толщина ребер (связей) между вершинами регулируется количеством совместного участия в отборах (при наведении это число показывается).

In [1]:
# https://visjs.github.io/vis-network/docs/network/physics.html

In [2]:
import pandas as pd, numpy as np, matplotlib as plt, seaborn as sns
from datetime import date, time, datetime
from sklearn import preprocessing
import pickle
from tqdm.auto import tqdm
from pathlib import Path
pd.set_option('display.float_format', lambda x: '%.0f' % x)
pd.set_option('display.max_columns', None)
from pyvis.network import Network
import warnings
warnings.filterwarnings('ignore')

In [3]:
as0 = pd.read_excel('ас1 анон.XLSX')

### В АС должны быть следующие столбцы: Лот ID, Поз.лота, Посл.версия, Поставщик, Поставщик.1, Создано: Дата, Создано: Время, Сумма без НДС, руб, Победитель

In [4]:
def graph_func(as1, all_filter=False):
    as1 = as1[as1['Поставщик'].notna()]
    as1['Создано'] = pd.to_datetime(as1['Создано: Дата'].astype(str) + ' ' + as1['Создано: Время'].astype(str),errors='coerce')
    as1['Победитель по цене'] = (as1['Лидер'].notna())&(as1['Посл.версия'].notna())
    if as1['Поставщик.1'].dtype == 'float64':
        k = as1.groupby('Поставщик.1').agg({'Поставщик':'nunique'}).reset_index()
        if len(k[k['Поставщик']>1])!=0:
            names = as1.groupby('Поставщик.1').agg({'Поставщик':'last'}).reset_index()
            as1 = as1.drop('Поставщик', axis=1)
            as1 = pd.merge(as1, names, how='left', on='Поставщик.1')
        as1 = as1.rename(columns={'Поставщик.1':'Код_поставщика', 'Поставщик':'Поставщик.1'})
        merged = pd.merge(as1, as1[as1['Победитель'] == 'X'][['Лот ID','Поз.лота','Поставщик.1', 'Сумма без НДС, руб']], how = 'left', on = ['Лот ID', 'Поз.лота'])
        merged['Победитель'] = np.where((merged['Победитель'] == 'X'), 1, 0)
    else:
        k = as1.groupby('Поставщик').agg({'Поставщик.1':'nunique'}).reset_index()
        if len(k[k['Поставщик.1']>1])!=0:
            names = as1.groupby('Поставщик').agg({'Поставщик.1':'last'}).reset_index()
            as1 = as1.drop('Поставщик.1', axis=1)
            as1 = pd.merge(as1, names, how='left', on='Поставщик')
        as1 = as1.rename(columns={'Поставщик':'Код_поставщика'})
        merged = pd.merge(as1, as1[as1['Победитель'] == 'X'][['Лот ID','Поз.лота','Поставщик.1', 'Сумма без НДС, руб']], how = 'left', on = ['Лот ID', 'Поз.лота'])
        merged['Победитель'] = np.where((merged['Победитель'] == 'X'), 1, 0)
# Считаем процент выигранных денег
    count = merged.groupby('Поставщик.1_x').agg({'Лот ID':'nunique'})
    count['Поз.лота'] = merged[merged['Посл.версия'] == 'X'].groupby(['Лот ID','Поз.лота','Поставщик.1_x']).agg({'Создано':'max'}).reset_index().groupby(['Поставщик.1_x']).agg({'Поз.лота':'count'})
    count['Сумма заявок'] = merged[merged['Посл.версия'] == 'X'].groupby(['Лот ID','Поз.лота','Поставщик.1_x']).agg({'Создано':'max', 'Сумма без НДС, руб_x':'first'}).reset_index().groupby(['Поставщик.1_x']).agg({'Сумма без НДС, руб_x':'sum'})
    count['Лидер позиции'] = merged[merged['Победитель по цене'] == 1].groupby(['Поставщик.1_x']).agg({'Поз.лота':'count'})
    count['Сумма лидирующих заявок'] = merged[merged['Победитель по цене'] == 1].groupby(['Поставщик.1_x']).agg({'Сумма без НДС, руб_x':'sum'})
    count['Количество поставленных лотов'] = merged[merged['Победитель']==1].groupby('Поставщик.1_y').agg({'Поз.лота':'count'})
    count['Сумма поставок'] = merged.groupby(['Лот ID', 'Поз.лота']).agg({'Поставщик.1_y':'first', 'Сумма без НДС, руб_y':'first'}).reset_index().groupby('Поставщик.1_y').agg({'Сумма без НДС, руб_y':'sum'})
    count['% выигранных денег'] = count['Сумма поставок'] / count['Сумма заявок']*100
    count.fillna(0, inplace=True)
    count = count.sort_values('Сумма поставок', ascending = False).reset_index()
    
    link = merged.groupby(['Лот ID','Поз.лота','Поставщик.1_x']).agg({'Победитель':'sum','Создано':'max', 'Поставщик.1_y':'first'}).reset_index().sort_values(by=['Лот ID','Поз.лота','Создано'], ascending=True)
    link = link.groupby(['Лот ID', 'Поз.лота', 'Поставщик.1_x']).agg({'Поставщик.1_y':'first'}).reset_index()
    link = link.groupby(['Поставщик.1_y', 'Поставщик.1_x']).agg({'Лот ID':'nunique'}).reset_index()
    link1 = pd.merge(link, count[['Поставщик.1_x', 'Сумма поставок']], how='left', left_on='Поставщик.1_y', right_on='Поставщик.1_x')
    link1 = link1.rename(columns={link1.columns[1]:'Поставщик.1_x'}).drop(link1.columns[3], axis=1)
    link2 = link1.copy()
    link2['Пара'] = link2[['Поставщик.1_y', 'Поставщик.1_x']].apply(sorted, axis=1).str.join('=')
# Группируем данные по парам поставщиков и суммируем значения 'Лот ID'
    link2 = link2.groupby('Пара')['Лот ID'].sum().reset_index()
    link2[['Поставщик.1_y', 'Поставщик.1_x']] = link2['Пара'].str.split('=', expand=True)
    link2 = link2.drop('Пара', axis=1)
    link3 = pd.merge(link2, link1, how='left', on=['Поставщик.1_y', 'Поставщик.1_x'])
    link3 = link3.rename(columns={link3.columns[0]:'Лот ID'}).drop(link3.columns[3], axis=1)
    link4 = link3[link3['Сумма поставок'].isna()]
    link5 = pd.merge(link4[['Лот ID', 'Поставщик.1_y', 'Поставщик.1_x']], link1, how='left', left_on=['Поставщик.1_y', 'Поставщик.1_x'], right_on=['Поставщик.1_x', 'Поставщик.1_y'])
    link5 = link5.drop(link5.columns[[0, 1, 2]], axis=1)
    link5 = link5.rename(columns={link5.columns[0]:'Поставщик.1_y', link5.columns[1]:'Поставщик.1_x', link5.columns[2]:'Лот ID'})
    link6 = pd.concat([link3[link3['Сумма поставок'].notna()], link5])
    link7 = link6.groupby('Поставщик.1_y').agg({'Поставщик.1_x':'count'}).reset_index()
    link7 = link7.rename(columns={'Поставщик.1_x':'Колво'})
    link8 = pd.merge(link6, link7, how='left', on='Поставщик.1_y')
    link8['Сум'] = link8['Сумма поставок'] / link8['Колво']
    link8 = pd.merge(link8, as1.groupby('Код_поставщика').agg({'Поставщик.1':'last'}).reset_index(), how='left', left_on='Поставщик.1_x', right_on='Поставщик.1')
    link8 = link8.drop('Поставщик.1', axis=1)
# Создаем граф, height можно менять значение для растягивания высоты 
    net = Network(height='720px', width='100%', bgcolor='#222222', font_color='white')
# Отбираем топ-10 худших по проценту выигранных денег компаний
    worst_comp = count[count['% выигранных денег'] > 0].nsmallest(10, '% выигранных денег')['Поставщик.1_x'].tolist()
    suppliers = set(link8['Поставщик.1_y']).union(set(link8['Поставщик.1_x']))
    top_comp = count.sort_values('Сумма поставок', ascending = False)['Поставщик.1_x'].iloc[:10].to_list()
    if all_filter:
        suppliers = [supplier for supplier in suppliers if link8[link8['Поставщик.1_y'] == supplier]['Сум'].sum() > 0]
# Создаем вершины, size - размер узла, color - цвет (красный для худших топ-20 по проценту побед, желтый - для топ-5 по объему) 
    for supplier in suppliers:
        size = link8[link8['Поставщик.1_y'] == supplier]['Сум'].sum()
        win = count[count['Поставщик.1_x'] == supplier]['% выигранных денег'].iloc[0]
        code = link8[link8['Поставщик.1_x'] == supplier]['Код_поставщика'].iloc[0]
        lot = count[count['Поставщик.1_x'] == supplier]['Лот ID'].iloc[0]
        pos1 = count[count['Поставщик.1_x'] == supplier]['Поз.лота'].iloc[0]
        pos2 = count[count['Поставщик.1_x'] == supplier]['Количество поставленных лотов'].iloc[0]
        lead = count[count['Поставщик.1_x'] == supplier]['Сумма лидирующих заявок'].iloc[0]
        orders = count[count['Поставщик.1_x'] == supplier]['Сумма заявок'].iloc[0]
# Устанавливаем цвет для худших по проценту побед и самых крупных поставщиков        
        color = 'yellow' if supplier in top_comp else 'red' if supplier in worst_comp else None
# Задаем масштаб для размеров узлов
        value = 11 if size<3**11 else np.log(size)/np.log(3)
        net.add_node(supplier, label=supplier, value=value, title=supplier+'\n'
                     +'Код: '+str(int(code))+'\n'
                     +'# лотов: '+str(int(lot))+'\n'
                     +'# позиций (% побед): '+str(int(pos2))+'/'+str(int(pos1))+' ('+str(int(int(pos2)/int(pos1)*100))+'%)'+'\n'
                     +'Сумма заявок: '+str(float(np.round(orders/1000000, 1)))+' млн руб'+'\n'
                     +'Сумма лидирующих заявок: '+str(float(np.round(lead/1000000, 1)))+' млн руб'+'\n'
                     +'Сумма поставок: '+str(float(np.round(size/1000000, 1)))+' млн руб'+'\n'
                     +'% выигранных денег: '+str(float(np.round(win, 2)))+'%', color=color)
# Создаем ребра, weight - толщина
    for i, row in link8[(link8['Поставщик.1_x']!=link8['Поставщик.1_y']) & 
                    (link8['Поставщик.1_x'].isin(suppliers)) & 
                    (link8['Поставщик.1_y'].isin(suppliers))].iterrows():
        source = row['Поставщик.1_y']
        target = row['Поставщик.1_x']
        weight = row['Лот ID']
        if row['Поставщик.1_x'] in worst_comp or row['Поставщик.1_y'] in worst_comp:
            color = 'red'
        elif row['Поставщик.1_x'] in top_comp or row['Поставщик.1_y'] in top_comp:
            color = 'yellow'
        else:
            color = None  # Цвет по умолчанию
        net.add_edge(source, target, value=weight, title=str(source)+'-'+str(target)+': '+str(weight), color=color)

# Параметры оформления
    net.set_options("""
    var options = {
        "nodes": {
            "opacity": 1,
            "font": {
                "color": "white"
            },
            "scaling":{
                "min": 6,
                "max": 60
            }
        },
        "physics": {
            "solver": "forceAtlas2Based",
            "stabilization": {
                "enabled": true
            },
            "forceAtlas2Based": {
                "damping": 1,
                "iterations": 1,
                "springConstant": 0.04,
                "onlyDynamicEdges": true
                },
            "minVelocity": 50
            },
        "edges": {
            "scaling": {
                "min": 0.5,
                "max": 20
            }
        }
        }
    """)
    net.show('graph.html', notebook=False)
# barnesHut, forceAtlas2Based - норм алгоритмы расположения узлов, также есть repulsion и hierarchicalRepulsion
# onlyDynamicEdges отвечает за стабилизацию графа
# springConstant определяет силу пружин - если узлы колбасит, нужно уменьшить значение этого параметра (базовое 0.08)
# minVelocity определяет скорость, при достижении которой всеми узлами граф стабилизируется (если поставить 50, то граф с самого начала будет статичным)

In [5]:
graph_func(as0)

graph.html


In [6]:
graph_func(as0, all_filter=True)

graph.html
